放置数据的文件夹格式如下：

![1.png](1.png)

训练代码如下：

In [ ]:
# 运行 main.py

# python main.py \
# --data_root ./datasets/selfData \
# --year 2012 \
# --model deeplabv3plus_resnet101 \
# --total_itrs 20000 \
# --crop_val \
# --lr 0.01 \
# --batch_size 16 \
# --crop_size 480 \
# --output_stride 8 \
# --num_classes 3 \
# --gpu_id 0,1,2,3

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
from docx import Document
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict

class _StreamMetrics(object):
    def __init__(self):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def update(self, gt, pred):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def get_results(self):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def to_str(self, metrics):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def reset(self):
        """ Overridden by subclasses """
        raise NotImplementedError()      

# 在 main.py 中调用 metrics = StreamSegMetrics(opts.num_classes)
class StreamSegMetrics(_StreamMetrics):
    """
    Stream Metrics for Semantic Segmentation Task
    """
    def __init__(self, n_classes):
        self.n_classes = n_classes
        # self.confusion_matrix = np.zeros((n_classes, n_classes))
        self.confusion_matrix = []
        self.tps = []
        self.fps = []
        self.ious = [[],[],[]]
        self.ious_result = [[],[],[]]
        # self.ious = np.empty((3, 0))
        self.boundary_ious = [[],[],[]]
        self.boundary_ious_result = [[],[],[]]
        self.min_ious = [[],[],[]]
        self.min_ious_result = [[],[],[]]

        # self.precision_list = []
        # TODO 这样就可以自动在数组里生成分类标签
        self.precision_list = defaultdict(list)
        # self.recall_list = []
        self.recall_list = defaultdict(list)
        self.iou_threshold = []

    def update(self, label_trues, label_preds):
        for lt, lp in zip(label_trues, label_preds):
            # self.confusion_matrix += self._fast_hist( lt.flatten(), lp.flatten() )

            hist = self._fast_hist( lt.flatten(), lp.flatten() )

            self.confusion_matrix.append(hist)
    
    @staticmethod
    def to_str(results):
        string = "\n"
        for k, v in results.items():
            if k!="Class IoU":
                string += "%s: %f\n"%(k, v)
        
        #string+='Class IoU:\n'
        #for k, v in results['Class IoU'].items():
        #    string += "\tclass %d: %f\n"%(k, v)
        return string
    
    # 只有当预测掩膜与真实掩膜之间的IoU大于或等于0.5时，该预测才被认为是正确的
    def calculate_iou(self, true_mask, pred_mask):
        """Calculate IoU for a single pair of masks."""
        intersection = np.logical_and(true_mask, pred_mask)
        union = np.logical_or(true_mask, pred_mask)
        iou = np.sum(intersection) / np.sum(union) if np.sum(union) > 0 else 0.0
        return iou

    def get_boundary(self, mask):
        # 可以打印出掩膜的形状吗
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boundary = np.zeros_like(mask)
        cv2.drawContours(boundary, contours, -1, 1, thickness=1)
        return boundary
    
    def calculate_boundary_iou(self, true_mask, pred_mask):
        pred_boundary = cv2.Canny(pred_mask.astype(np.uint8) * 255, 50, 150)
        true_boundary = cv2.Canny(true_mask.astype(np.uint8) * 255, 50, 150)

        # 膨胀操作
        kernel = np.ones((3, 3), np.uint8)
        pred_boundary = cv2.dilate(pred_boundary, kernel) # boundary iou 的计算好像不准确
        true_boundary = cv2.dilate(true_boundary, kernel)

        # 计算交集
        intersection = np.sum((pred_boundary & true_boundary) > 0)

        # 计算并集
        union = np.sum((pred_boundary | true_boundary) > 0)

        # 计算 Boundary IoU
        if union == 0:
            return 0.0  # 避免除以零
        iou = intersection / union
        return iou

    
    def calculate_metrics(self, true_masks, pred_masks, num_classes):
        for class_id in range(3):
            true_mask = (true_masks == class_id).astype(np.uint8)
            pred_mask = (pred_masks == class_id).astype(np.uint8)

            iou = self.calculate_iou(true_mask, pred_mask)
            boundary_iou = self.calculate_boundary_iou(true_mask, pred_mask)

            # 将所有类别的都整合到一起
            self.ious[class_id].append(iou)
            self.ious_result[class_id] = sum(self.ious[class_id])/len(self.ious[class_id])

            self.boundary_ious[class_id].append(boundary_iou)
            self.boundary_ious_result[class_id] = sum(self.boundary_ious[class_id])/len(self.boundary_ious[class_id])

            min_iou = min(iou, boundary_iou)
            self.min_ious[class_id].append(min_iou)
            self.min_ious_result[class_id] = sum(self.min_ious[class_id])/len(self.min_ious[class_id])

        np.set_printoptions(suppress=True)

        # return ious, boundary_ious, min_ious


    def _fast_hist(self, label_true, label_pred):
        # label_true为numpy形式的标签数组。len(label_true)为230400，原图像为 480*480 = 230400
        # 当mask中的某个元素为True时，它表示label_true中对应位置的元素应该被选中；当为False时，则不应该被选中
        mask = (label_true >= 0) & (label_true < self.n_classes)
        # 根据掩码计算出大小
        areas = np.sum(label_true)
        # print('areas', areas)
        hist = np.bincount(
                self.n_classes * label_true[mask].astype(int) + label_pred[mask], # 将二维的（真实标签，预测标签）对映射到一维的索引上，其中每个索引都唯一地对应于一个（真实类别，预测类别）的组合。
                minlength=self.n_classes ** 2,
            ).reshape(self.n_classes, self.n_classes)

        if areas < 10000:
            # iu = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist)) # 计算每个矩阵各自的 iou 值 格式如：[0.84736547 0.28503124 0.11428243]
            # print('iu +++', iu)
            TPs = np.diag(hist)
            # 计算每个类别的 IoU
            # ious = []
            for i in range(len(TPs)):
                intersection = TPs[i]
                union = np.sum(hist[i, :]) + np.sum(hist[:, i]) - intersection
                if union > 0:
                    IoU = intersection / union
                    self.ious[i].append(IoU)
                    self.ious_result[i] = sum(self.ious[i])/len(self.ious[i])
                    iou_thresholds = np.arange(0.5, 0.95, 0.05)
                    for iou_threshold in iou_thresholds:
                        if IoU > iou_threshold:
                            TP = hist[i, i]
                            FP = np.sum(hist[:, i]) - TP
                            FN = np.sum(hist[i, :]) - TP
                                
                            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
                            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
                                
                            self.precision_list[iou_threshold].append(precision)
                            self.recall_list[iou_threshold].append(recall)
            return hist
        else:
            return hist


    def get_results(self):
        """Returns accuracy score evaluation result.
            - overall accuracy
            - mean accuracy
            - mean IU
            - fwavacc
        """

        # 一个考虑背景，一个不考虑。。。
        hists = self.confusion_matrix # self.confusion_matrix += self._fast_hist( lt.flatten(), lp.flatten() ) 

        combine_hist = np.zeros((self.n_classes, self.n_classes))

        aplist = []

        # iou_threshold = []
        for hist in hists:
            combine_hist += hist
            iu = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist)) # 格式如：[0.84736547 0.28503124 0.11428243]

        # precision  = np.array(self.precision_list)
        # recall  = np.array(self.recall_list)

        # 计算 AP 值
        for iou_threshold in self.precision_list:
            combined = list(zip(self.recall_list[iou_threshold], self.precision_list[iou_threshold]))
            combined_sorted = sorted(combined, key=lambda x: x[0])

            recall_sorted = np.array([r for r, p in combined_sorted])
            precision_sorted = np.array([p for r, p in combined_sorted])

            m = np.zeros_like(recall_sorted)
            for i in range(len(precision_sorted)):
                m[i] = np.max(precision_sorted[recall_sorted >= recall_sorted[i]])
            # 计算 AP
            ap = np.sum((recall_sorted[1:] - recall_sorted[:-1]) * m[1:])
            print('ap === ', ap)
            aplist.append(ap)

            # plt.figure(figsize=(8, 6))
            plt.plot(recall_sorted, precision_sorted, marker='o', linestyle='-', color='b')

            # 设置标题和标签
            plt.title('Precision-Recall Curve')
            plt.xlabel('Recall')
            plt.ylabel('Precision')

            # 添加网格
            plt.grid()

            # 显示图形
            plt.savefig('Precision-Recall Curve')

        aplist = np.array(aplist)
        mAP = np.nanmean(aplist)
        print('mAP', mAP)

        acc = np.diag(combine_hist).sum() / combine_hist.sum()
        acc_cls = np.diag(combine_hist) / combine_hist.sum(axis=1) 
        cls_acc = dict(zip(range(self.n_classes), acc_cls))
        print('cls_acc', cls_acc) # 每个类别准确率
        acc_cls = np.nanmean(acc_cls)

        # 计算 iou 时，要经过阈值的筛选 
        iu = np.diag(combine_hist) / (combine_hist.sum(axis=1) + combine_hist.sum(axis=0) - np.diag(combine_hist)) 
        print('iu', iu)

        mean_iu = np.nanmean(iu) 
        freq = combine_hist.sum(axis=1) / combine_hist.sum() 
        fwavacc = (freq[freq > 0] * iu[freq > 0]).sum() 

        cls_iu = dict(zip(range(self.n_classes), iu)) # 每个类别交并比

        ious = self.ious_result #

        print('ious_result', ious)

        return {
                "Overall Acc": acc,
                "Mean Acc": acc_cls,
                "FreqW Acc": fwavacc,
                "Mean IoU": mean_iu,
                # "precision": precision,
                # "recall": recall,
                "Class IoU": cls_iu,
            }
        
    def reset(self):
        # self.confusion_matrix = np.zeros((self.n_classes, self.n_classes))
        self.confusion_matrix = []

class AverageMeter(object):
    """Computes average values"""
    def __init__(self):
        self.book = dict()

    def reset_all(self):
        self.book.clear()
    
    def reset(self, id):
        item = self.book.get(id, None)
        if item is not None:
            item[0] = 0
            item[1] = 0

    def update(self, id, val):
        record = self.book.get(id, None)
        if record is None:
            self.book[id] = [val, 1]
        else:
            record[0]+=val
            record[1]+=1

    def get_results(self, id):
        record = self.book.get(id, None)
        assert record is not None
        return record[0] / record[1]
